# Python Logging Module: Complete Tutorial

**From Beginner to Advanced**

This notebook will take you through:
1. Why logging matters
2. Basic configuration and log levels
3. Handlers for different outputs (console, file, rotating)
4. Formatters and Filters for control
5. Configuration methods (dictConfig, logging.conf)
6. Custom handlers for domain-specific logging
7. Async logging for production systems
8. Best practices and real-world patterns

---

## Section 1: Introduction - Why Logging Matters

### The Problem
When building real applications, you need to understand what's happening at runtime:
- **Print statements**: Ad-hoc, no timestamps, can't be disabled, goes everywhere
- **Logging**: Professional, configurable, structured, can be controlled at runtime

### Print vs Logging Example

In [15]:
import logging

# --- Using Print Statements (Poor) ---
print("Starting process")
print("User authenticated")
print("Database connected")
print("Error: Connection failed")  # No severity level
print("Process complete")

Starting process
User authenticated
Database connected
Error: Connection failed
Process complete


In [16]:
# --- Using Logging (Professional) ---
import logging

# Reset logging for this cell (remove existing handlers)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Configure logging (one-time setup)
logging.basicConfig(
    level=logging.DEBUG, format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

logger.debug("Starting process")
logger.info("User authenticated")
logger.info("Database connected")
logger.error("Error: Connection failed")  # Severity level specified
logger.info("Process complete")

2026-06-12 18:44:17,436 - DEBUG - Starting process
2026-06-12 18:44:17,437 - INFO - User authenticated
2026-06-12 18:44:17,437 - INFO - Database connected
2026-06-12 18:44:17,437 - ERROR - Error: Connection failed
2026-06-12 18:44:17,437 - INFO - Process complete


### Key Advantages of Logging
- **Timestamps**: Know exactly when events occurred
- **Severity levels**: DEBUG, INFO, WARNING, ERROR, CRITICAL
- **Disable/enable at runtime**: No code changes needed
- **Multiple destinations**: Console, file, network, email, etc.
- **Formatting control**: Customize what gets logged and how
- **Production-ready**: Built into Python standard library

## Section 2: Basic Example with basicConfig

The simplest way to get started: `logging.basicConfig()`

**Important**: `basicConfig()` only works before creating loggers. Call it first!

In [17]:
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Configure logging (do this first, before any logger usage)
logging.basicConfig(
    level=logging.DEBUG,  # Minimum level to capture
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)

# Get a logger
logger = logging.getLogger("my_app")

# Log at different levels
logger.debug("Debug message - detailed diagnostic info")
logger.info("Info message - general informational message")
logger.warning("Warning message - something unexpected happened")
logger.error("Error message - a serious problem occurred")
logger.critical("Critical message - system failure imminent")

2026-06-12 18:44:31,062 - my_app - DEBUG - Debug message - detailed diagnostic info
2026-06-12 18:44:31,062 - my_app - INFO - Info message - general informational message
2026-06-12 18:44:31,063 - my_app - WARNING - Warning message - something unexpected happened
2026-06-12 18:44:31,063 - my_app - ERROR - Error message - a serious problem occurred
2026-06-12 18:44:31,064 - my_app - CRITICAL - Critical message - system failure imminent


## Section 3: Understanding Log Levels

Log levels control which messages are captured:

| Level | Value | Use Case |
|-------|-------|----------|
| DEBUG | 10 | Detailed diagnostic info for developers |
| INFO | 20 | General information about application flow |
| WARNING | 30 | Warning about potential issues (default level) |
| ERROR | 40 | Error occurred, functionality broken |
| CRITICAL | 50 | Serious error, system may not continue |
| NOTSET | 0 | No level set (inherit from parent) |

In [18]:
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Example 1: Setting level to WARNING (only WARNING and above are logged)
logging.basicConfig(level=logging.WARNING, format="%(levelname)s - %(message)s")

logger = logging.getLogger("app")
logger.debug("This is NOT shown (DEBUG < WARNING)")
logger.info("This is NOT shown (INFO < WARNING)")
logger.warning("This IS shown")
logger.error("This IS shown")
logger.critical("This IS shown")

WARNING - This IS shown
ERROR - This IS shown
CRITICAL - This IS shown


In [21]:
# Example 2: Changing log level at runtime
import logging
import sys

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Send logging to stdout so ordering matches print() in notebooks
logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s - %(message)s",
    stream=sys.stdout,
)
logger = logging.getLogger("app2")

# Reset this named logger's state so reruns behave consistently in notebooks
logger.handlers.clear()
logger.propagate = True
logger.setLevel(logging.DEBUG)

print("--- Current level: DEBUG ---", flush=True)
logger.debug("Debug message")
logger.info("Info message")

# Change level at runtime
logger.setLevel(logging.ERROR)
print("\n--- After changing to ERROR ---", flush=True)
logger.debug("Debug message (not shown)")
logger.info("Info message (not shown)")
logger.error("Error message")

--- Current level: DEBUG ---
DEBUG - Debug message
INFO - Info message

--- After changing to ERROR ---
ERROR - Error message


## Section 4: Loggers and basicConfig in Detail

### Key Components
- **Logger**: The interface you use to log messages
- **Handler**: Determines where logs go (console, file, etc.)
- **Formatter**: Controls the format of log messages
- **Filter**: Filters which records are logged

### Logger Hierarchy
Loggers are organized hierarchically by name (like modules):
- `root` - Top level
- `app` - Child of root
- `app.database` - Child of app
- `app.database.connection` - Child of app.database

In [22]:
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Configure root logger
logging.basicConfig(
    level=logging.DEBUG, format="%(name)s - %(levelname)s - %(message)s"
)

# Different loggers
root_logger = logging.getLogger()  # Root logger
app_logger = logging.getLogger("app")
db_logger = logging.getLogger("app.database")
conn_logger = logging.getLogger("app.database.connection")

root_logger.info("Message from root")
app_logger.info("Message from app")
db_logger.info("Message from app.database")
conn_logger.info("Message from app.database.connection")

root - INFO - Message from root
app - INFO - Message from app
app.database - INFO - Message from app.database
app.database.connection - INFO - Message from app.database.connection


In [24]:
# Logger hierarchy in action: changing parent logger level
import logging
import sys

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Send logging to stdout so ordering matches print() in notebooks
logging.basicConfig(
    level=logging.DEBUG,
    format="%(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
)

app_logger = logging.getLogger("app")
db_logger = logging.getLogger("app.database")

# Reset named logger state so reruns are consistent
app_logger.handlers.clear()
db_logger.handlers.clear()
app_logger.propagate = True
db_logger.propagate = True
app_logger.setLevel(logging.NOTSET)
db_logger.setLevel(logging.NOTSET)

print("--- Before: both log DEBUG ---", flush=True)
app_logger.debug("App debug")
db_logger.debug("DB debug")

# Change parent logger level
app_logger.setLevel(logging.WARNING)
print("\n--- After setting app to WARNING ---", flush=True)
app_logger.debug("App debug (not shown)")
db_logger.debug("DB debug (not shown, inherits from parent)")

--- Before: both log DEBUG ---
app - DEBUG - App debug
app.database - DEBUG - DB debug

--- After setting app to WARNING ---


## Section 5: StreamHandler - Console Logging

`StreamHandler` sends logs to a stream (usually stdout or stderr).

In [25]:
import logging
import sys

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Create logger
logger = logging.getLogger("stream_example")
logger.setLevel(logging.DEBUG)

# Create StreamHandler (outputs to console)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)

# Create formatter
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

# Add handler to logger
logger.addHandler(handler)

# Log messages
logger.debug("Debug message")
logger.info("Info message")
logger.warning("Warning message")

2026-06-12 19:00:10,379 - stream_example - DEBUG - Debug message
2026-06-12 19:00:10,380 - stream_example - INFO - Info message
2026-06-12 19:00:10,380 - stream_example - WARNING - Warning message


In [27]:
# Example: Different formatters for different handlers
import logging
import sys

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

logger = logging.getLogger("multi_handler")
logger.setLevel(logging.DEBUG)

# Reset named logger state so reruns are consistent
logger.handlers.clear()
logger.propagate = False

# Handler 1: Simple format, INFO+
handler1 = logging.StreamHandler(sys.stdout)
handler1.setLevel(logging.INFO)
formatter1 = logging.Formatter("SIMPLE | %(levelname)s - %(message)s")
handler1.setFormatter(formatter1)

# Handler 2: Detailed format, WARNING+
handler2 = logging.StreamHandler(sys.stdout)
handler2.setLevel(logging.WARNING)
formatter2 = logging.Formatter(
    "DETAILED | %(asctime)s | %(name)s | %(levelname)s | %(message)s"
)
handler2.setFormatter(formatter2)

logger.addHandler(handler1)
logger.addHandler(handler2)

logger.debug("Debug (not shown by either handler)")
logger.info("Info message (shown once by SIMPLE handler)")
logger.warning("Warning message (shown by both handlers with different formats)")

SIMPLE | INFO - Info message (shown once by SIMPLE handler)
SIMPLE | WARNING - Warning message (shown by both handlers with different formats)
DETAILED | 2026-06-12 19:04:40,240 | multi_handler | WARNING | Warning message (shown by both handlers with different formats)


In [28]:
# Example: Emit using only one chosen handler
import logging
import sys

# Fresh setup for this demonstration
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
logging.root.handlers = []

logger = logging.getLogger("single_handler_demo")
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
logger.propagate = False

simple_handler = logging.StreamHandler(sys.stdout)
simple_handler.setLevel(logging.INFO)
simple_handler.setFormatter(logging.Formatter("SIMPLE | %(levelname)s - %(message)s"))

detailed_handler = logging.StreamHandler(sys.stdout)
detailed_handler.setLevel(logging.INFO)
detailed_handler.setFormatter(
    logging.Formatter("DETAILED | %(asctime)s | %(name)s | %(levelname)s | %(message)s")
)

logger.addHandler(simple_handler)
logger.addHandler(detailed_handler)

print("Normal logger.info -> both handlers run:", flush=True)
logger.info("This appears twice (once per handler)")


# Helper to send one record to exactly one handler
def emit_with_one_handler(target_handler, target_logger, level, message):
    record = target_logger.makeRecord(
        name=target_logger.name,
        level=level,
        fn="<notebook>",
        lno=0,
        msg=message,
        args=(),
        exc_info=None,
    )
    target_handler.handle(record)


print("\nOnly SIMPLE handler:", flush=True)
emit_with_one_handler(simple_handler, logger, logging.INFO, "Printed with SIMPLE only")

print("\nOnly DETAILED handler:", flush=True)
emit_with_one_handler(
    detailed_handler, logger, logging.INFO, "Printed with DETAILED only"
)

Normal logger.info -> both handlers run:
SIMPLE | INFO - This appears twice (once per handler)
DETAILED | 2026-06-12 19:08:28,297 | single_handler_demo | INFO | This appears twice (once per handler)

Only SIMPLE handler:
SIMPLE | INFO - Printed with SIMPLE only

Only DETAILED handler:
DETAILED | 2026-06-12 19:08:28,299 | single_handler_demo | INFO | Printed with DETAILED only


## Section 6: FileHandler - File Logging

`FileHandler` writes logs to a file.

In [1]:
import logging
import os

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Create logs directory if it doesn't exist
os.makedirs("logs", exist_ok=True)

# Create logger
logger = logging.getLogger("file_example")
logger.setLevel(logging.DEBUG)

# Create FileHandler
file_handler = logging.FileHandler("logs/app.log")
file_handler.setLevel(logging.DEBUG)

# Create formatter
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)

# Add handler to logger
logger.addHandler(file_handler)

# Log messages
logger.debug("Debug message written to file")
logger.info("Info message written to file")
logger.error("Error message written to file")

print("Logs written to logs/app.log")

Logs written to logs/app.log


In [2]:
# Read and display the log file
with open("logs/app.log", "r") as f:
    content = f.read()
    print("Contents of logs/app.log:")
    print(content)

Contents of logs/app.log:
2026-06-18 18:28:57,556 - file_example - DEBUG - Debug message written to file
2026-06-18 18:28:57,556 - file_example - INFO - Info message written to file
2026-06-18 18:28:57,556 - file_example - ERROR - Error message written to file



## Section 7: RotatingFileHandler - Rotation and Backups

For production applications, log files can grow very large. `RotatingFileHandler` rotates logs based on size.

In [ ]:
import logging
from logging.handlers import RotatingFileHandler
import os

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

os.makedirs("logs", exist_ok=True)

# Create logger
logger = logging.getLogger("rotating_example")
logger.setLevel(logging.DEBUG)

# Create RotatingFileHandler
# maxBytes: rotate when file reaches 5KB
# backupCount: keep 3 backup files
rotating_handler = RotatingFileHandler(
    "logs/rotating_app.log",
    maxBytes=5000,  # 5 KB
    backupCount=3,  # Keep 3 backups
)
rotating_handler.setLevel(logging.DEBUG)

# Create formatter
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
rotating_handler.setFormatter(formatter)

# Add handler
logger.addHandler(rotating_handler)

# Generate logs to trigger rotation
for i in range(50):
    logger.info(
        f"Log message number {i} - This is a sample log entry to demonstrate rotation"
    )

print("Generated 50 log messages")

In [ ]:
# Check what log files were created
import os

log_files = [f for f in os.listdir("logs") if f.startswith("rotating_app")]
print(f"Log files created: {sorted(log_files)}")

# Show sizes
for log_file in sorted(log_files):
    size = os.path.getsize(f"logs/{log_file}")
    print(f"{log_file}: {size} bytes")

### Time-Based Rotation

You can also rotate logs based on time using `TimedRotatingFileHandler`.

In [ ]:
import logging
from logging.handlers import TimedRotatingFileHandler
import os

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

os.makedirs("logs", exist_ok=True)

# Create logger
logger = logging.getLogger("time_rotating_example")
logger.setLevel(logging.DEBUG)

# Rotate daily at midnight
time_handler = TimedRotatingFileHandler(
    "logs/daily_app.log",
    when="midnight",  # Rotate at midnight
    interval=1,  # Every 1 day
    backupCount=7,  # Keep 7 days of logs
)
time_handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
time_handler.setFormatter(formatter)

logger.addHandler(time_handler)

# Log a message
logger.info("This log will rotate daily at midnight")
print("Time-based rotation configured for daily rotation at midnight")

## Section 8: Formatters - Customizing Log Format

Format strings control what information appears in each log message.

### Common Format Attributes
- `%(asctime)s` - Timestamp of log message
- `%(name)s` - Logger name
- `%(levelname)s` - Level (DEBUG, INFO, WARNING, ERROR, CRITICAL)
- `%(message)s` - The actual log message
- `%(filename)s` - Source filename
- `%(lineno)d` - Source line number
- `%(funcName)s` - Function name
- `%(pathname)s` - Full file path
- `%(process)d` - Process ID
- `%(thread)d` - Thread ID

In [ ]:
# Example 3: Custom timestamp format
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

formatter = logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",  # Custom date format
)

logger = logging.getLogger("custom_timestamp")
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Message with custom timestamp format")

In [ ]:
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []


# Filter that only allows INFO and above
class MinLevelFilter(logging.Filter):
    def __init__(self, level):
        super().__init__()
        self.level = level

    def filter(self, record):
        return record.levelno >= self.level


# Setup
logger = logging.getLogger("filter_example")
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
formatter = logging.Formatter("%(levelname)s - %(message)s")
handler.setFormatter(formatter)

# Add filter
handler.addFilter(MinLevelFilter(logging.INFO))
logger.addHandler(handler)

print("--- With MinLevelFilter(INFO) ---")
logger.debug("Debug (filtered out)")
logger.info("Info (shown)")
logger.warning("Warning (shown)")

In [ ]:
# Example 2: Filter by logger name
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []


class LoggerNameFilter(logging.Filter):
    """Only allow logs from specific logger names"""

    def __init__(self, allowed_names):
        super().__init__()
        self.allowed_names = allowed_names

    def filter(self, record):
        for name in self.allowed_names:
            if record.name.startswith(name):
                return True
        return False


# Setup
handler = logging.StreamHandler()
formatter = logging.Formatter("%(name)s - %(message)s")
handler.setFormatter(formatter)
handler.addFilter(LoggerNameFilter(["app.auth", "app.database"]))

# Create multiple loggers
auth_logger = logging.getLogger("app.auth")
db_logger = logging.getLogger("app.database")
other_logger = logging.getLogger("app.utils")

for logger in [auth_logger, db_logger, other_logger]:
    logger.setLevel(logging.DEBUG)
    logger.addHandler(handler)

print("--- With LoggerNameFilter (['app.auth', 'app.database']) ---")
auth_logger.info("Auth message (shown)")
db_logger.info("DB message (shown)")
other_logger.info("Utils message (filtered out)")

## Section 9: Filters - Selective Logging

Filters allow you to selectively log records based on custom logic.

In [ ]:
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

logging.basicConfig(
    level=logging.DEBUG, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("format_example")
logger.info("Basic format example")

In [ ]:
# Example 2: More detailed format with file info
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

logging.basicConfig(
    level=logging.DEBUG,
    format="[%(asctime)s] %(levelname)-8s [%(filename)s:%(lineno)d] %(message)s",
)

logger = logging.getLogger("detailed")
logger.debug("Debug with detailed format")
logger.info("Info with detailed format")
logger.error("Error with detailed format")

## Section 10: Configuration Methods - dictConfig and logging.conf

For complex applications, manage logging configuration externally instead of hardcoding.

### Method 1: Dictionary Configuration (dictConfig)

In [ ]:
import logging
import logging.config
import os

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

os.makedirs("logs", exist_ok=True)

# Define configuration as a dictionary
LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "standard": {"format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s"},
        "detailed": {
            "format": "[%(asctime)s] %(levelname)-8s [%(filename)s:%(lineno)d] %(message)s"
        },
    },
    "handlers": {
        "console": {
            "class": "logging.StreamHandler",
            "level": "DEBUG",
            "formatter": "standard",
            "stream": "ext://sys.stdout",
        },
        "file": {
            "class": "logging.FileHandler",
            "level": "DEBUG",
            "formatter": "detailed",
            "filename": "logs/dictconfig_app.log",
        },
    },
    "loggers": {
        "app": {"level": "DEBUG", "handlers": ["console", "file"]},
        "app.database": {"level": "WARNING", "handlers": ["console"]},
    },
}

# Apply configuration
logging.config.dictConfig(LOGGING_CONFIG)

# Get loggers
app_logger = logging.getLogger("app")
db_logger = logging.getLogger("app.database")

# Log messages
app_logger.info("Application started")
db_logger.debug("DB debug (not shown, level is WARNING)")
db_logger.warning("DB warning")

In [ ]:
# Verify file output
print("\nContents of logs/dictconfig_app.log:")
with open("logs/dictconfig_app.log", "r") as f:
    print(f.read())

### Method 2: INI File Configuration (logging.conf)

In [ ]:
# First, create a logging config file
import logging
import os

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

os.makedirs("logs", exist_ok=True)

config_content = """[loggers]
keys=root,app,app.database

[handlers]
keys=console,file

[formatters]
keys=standard,detailed

[logger_root]
level=DEBUG
handlers=console

[logger_app]
level=DEBUG
handlers=console,file
qualname=app
propagate=0

[logger_app.database]
level=WARNING
handlers=console
qualname=app.database
propagate=0

[handler_console]
class=StreamHandler
level=DEBUG
formatter=standard
args=(sys.stdout,)

[handler_file]
class=FileHandler
level=DEBUG
formatter=detailed
args=('logs/ini_config_app.log',)

[formatter_standard]
format=%(asctime)s - %(name)s - %(levelname)s - %(message)s

[formatter_detailed]
format=[%(asctime)s] %(levelname)-8s [%(filename)s:%(lineno)d] %(message)s
"""

with open("logs/logging.conf", "w") as f:
    f.write(config_content)

print("Created logging.conf")

In [ ]:
# Load configuration from INI file
import logging
import logging.config

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

logging.config.fileConfig("logs/logging.conf")

# Get loggers
app_logger = logging.getLogger("app")
db_logger = logging.getLogger("app.database")

# Log messages
app_logger.info("Application started from INI config")
db_logger.debug("DB debug (not shown)")
db_logger.warning("DB warning")

print("Logging configured from INI file")

## Section 11: Custom Handlers - Domain-Specific Logging

Create custom handlers for domain-specific logging needs (e.g., email, Slack, database).

In [ ]:
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []


# Custom handler that logs to an in-memory list
class MemoryHandler(logging.Handler):
    """Custom handler that stores logs in memory"""

    def __init__(self, max_records=100):
        super().__init__()
        self.records = []
        self.max_records = max_records

    def emit(self, record):
        """Store the log record"""
        if len(self.records) >= self.max_records:
            self.records.pop(0)  # Remove oldest
        self.records.append(self.format(record))

    def get_records(self):
        """Retrieve stored records"""
        return self.records


# Setup
logger = logging.getLogger("memory_example")
logger.setLevel(logging.DEBUG)

# Add custom handler
memory_handler = MemoryHandler()
formatter = logging.Formatter("%(levelname)s - %(message)s")
memory_handler.setFormatter(formatter)
logger.addHandler(memory_handler)

# Log messages
logger.info("First message")
logger.warning("Second message")
logger.error("Third message")

# Retrieve from memory
print("\nLogs in memory:")
for record in memory_handler.get_records():
    print(f"  {record}")

In [ ]:
# Example 2: Custom handler that only logs errors
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []


class ErrorOnlyHandler(logging.Handler):
    """Custom handler that only processes ERROR and CRITICAL messages"""

    def __init__(self):
        super().__init__()
        self.errors = []

    def emit(self, record):
        if record.levelno >= logging.ERROR:
            self.errors.append({
                "level": record.levelname,
                "message": record.getMessage(),
                "timestamp": record.created,
            })

    def get_errors(self):
        return self.errors


# Setup
logger = logging.getLogger("error_tracking")
logger.setLevel(logging.DEBUG)

error_handler = ErrorOnlyHandler()
logger.addHandler(error_handler)

# Log messages
logger.debug("Debug message")
logger.info("Info message")
logger.warning("Warning message")
logger.error("Error message 1")
logger.critical("Critical message")
logger.error("Error message 2")

# Retrieve errors
print("\nErrors captured:")
for error in error_handler.get_errors():
    print(f"  [{error['level']}] {error['message']}")

## Section 12: Async Logging - Non-Blocking Logs

For high-performance applications, use async logging to avoid blocking on I/O.

In [ ]:
import logging
from logging.handlers import QueueHandler, QueueListener
import os
import queue

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

os.makedirs("logs", exist_ok=True)

# Create a queue
log_queue = queue.Queue()

# Setup logger with QueueHandler (non-blocking)
logger = logging.getLogger("async_example")
logger.setLevel(logging.DEBUG)

queue_handler = QueueHandler(log_queue)
logger.addHandler(queue_handler)

# Setup listener with FileHandler (blocking)
file_handler = logging.FileHandler("logs/async_app.log")
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)

# QueueListener processes logs from queue
listener = QueueListener(log_queue, file_handler, respect_handler_level=True)
listener.start()

# Log messages (non-blocking)
print("Logging 10 messages asynchronously...")
for i in range(10):
    logger.info(f"Async log message {i}")

# Stop listener
listener.stop()

print("Async logging complete")
print("\nFirst few lines from logs/async_app.log:")
with open("logs/async_app.log", "r") as f:
    lines = f.readlines()[:5]
    for line in lines:
        print(line.rstrip())

## Section 12B: Best Practices and Real-World Patterns

### Best Practices Summary

In [ ]:
# Best Practice 1: Use __name__ for logger name
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

# Good: Logger name matches module name
logger = logging.getLogger(__name__)

# This way, you can easily manage loggers by module hierarchy
print(f"Logger name: {logger.name}")

# Best Practice 2: Configure logging once at application startup
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)
logger.info("Application started")

In [ ]:
# Best Practice 3: Avoid logging sensitive information
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
formatter = logging.Formatter("%(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

# Good: Mask sensitive data
user_id = "12345"
logger.info(f"User authenticated: user_id={user_id[-4:]}***")  # Show only last 4 digits

# Bad (commented out to avoid showing secrets):
# logger.info(f"Password: {password}")  # Never log passwords
# logger.info(f"API Key: {api_key}")    # Never log secrets

In [ ]:
# Best Practice 4: Use appropriate log levels
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
formatter = logging.Formatter("%(levelname)s - %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

# DEBUG: Development diagnostics
logger.debug("Variable x = 42")

# INFO: Application milestones
logger.info("Application started successfully")

# WARNING: Potential issues
logger.warning("API response time exceeded threshold")

# ERROR: Something went wrong
logger.error("Failed to connect to database")

# CRITICAL: System failure
logger.critical("Out of memory - system shutting down")

In [ ]:
# Best Practice 5: Include context in logs (use extra parameter)
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []


class ContextFilter(logging.Filter):
    def filter(self, record):
        # Add request_id to all log records
        if not hasattr(record, "request_id"):
            record.request_id = "N/A"
        return True


logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
formatter = logging.Formatter("[%(request_id)s] %(message)s")
handler.setFormatter(formatter)
handler.addFilter(ContextFilter())
logger.addHandler(handler)

# Log with context
logger.info("User action", extra={"request_id": "REQ-12345"})
logger.info("Another action", extra={"request_id": "REQ-12346"})

In [ ]:
# Best Practice 6: Exception logging
import logging

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler()
formatter = logging.Formatter("%(levelname)s - %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

try:
    result = 1 / 0  # Divide by zero
except ZeroDivisionError:
    # Good: Use exc_info=True to include traceback
    logger.error("Mathematical error occurred", exc_info=True)
    # Or use exception() shortcut
    # logger.exception("Mathematical error occurred")

### Complete Production-Ready Example

In [ ]:
import logging
import logging.config
import os

# Reset logging for this cell
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
logging.root.handlers = []

os.makedirs("logs", exist_ok=True)

# Production-ready configuration
LOGGING_CONFIG = {
    "version": 1,
    "disable_existing_loggers": False,
    "formatters": {
        "verbose": {
            "format": "{levelname} {asctime} {name} {process:d} {thread:d} {message}",
            "style": "{",
        },
        "simple": {
            "format": "{levelname} {message}",
            "style": "{",
        },
    },
    "filters": {
        "require_debug_true": {
            "()": "logging.Filter",
        },
    },
    "handlers": {
        "console": {
            "level": "INFO",
            "class": "logging.StreamHandler",
            "formatter": "simple",
        },
        "file": {
            "level": "DEBUG",
            "class": "logging.handlers.RotatingFileHandler",
            "formatter": "verbose",
            "filename": "logs/production.log",
            "maxBytes": 1048576,  # 1MB
            "backupCount": 5,
        },
    },
    "root": {
        "level": "DEBUG",
        "handlers": ["console", "file"],
    },
    "loggers": {
        "app.database": {
            "level": "INFO",
            "handlers": ["file"],
            "propagate": False,
        },
        "app.auth": {
            "level": "DEBUG",
            "handlers": ["console", "file"],
            "propagate": False,
        },
    },
}

# Apply configuration
logging.config.dictConfig(LOGGING_CONFIG)

# Use loggers
logger = logging.getLogger(__name__)
auth_logger = logging.getLogger("app.auth")
db_logger = logging.getLogger("app.database")

logger.info("Production logging configured")
auth_logger.debug("Auth module initialized")
db_logger.info("Database module initialized")

print("Production-ready logging setup complete!")

## Summary

You now understand:
- ✅ Why logging is better than print statements
- ✅ Log levels and how to use them
- ✅ Handlers for console, file, and rotating logs
- ✅ Formatters for custom output
- ✅ Filters for selective logging
- ✅ Configuration methods (basicConfig, dictConfig, INI files)
- ✅ Custom handlers for domain-specific needs
- ✅ Async logging for performance
- ✅ Best practices for production systems

**Next Steps**:
1. Read the `README.md` for a quick reference
2. Try modifying the examples to suit your use cases
3. Integrate logging into your own projects
4. Experiment with custom handlers and filters